# Notebook 1 — Did the tariff reduce peak residential load?

**Headline question for a utility:** A small peak-shaving tariff pilot treated five homes. Should we roll it out — and how much evening peak did it actually cut?

This notebook answers that with synthetic control, then stress-tests the answer:

1. **Question** — peak kW avoided, not a forecast
2. **Method** — synthetic control vs naive before/after
3. **Counterfactual** — treated path vs donor-weighted `Y(0)`
4. **Effect ± uncertainty** — ATT in kW and %, with a bootstrap interval
5. **Placebos** — untreated homes and fake intervention dates
6. **Cross-checks** — DiD and a propensity-weighted estimate
7. **Robustness** — donor pool, pre-period length, recovery of the injected truth
8. **Decision** — what this means for rollout, and what would break on real meters

See also: [`docs/causal_mental_model.md`](../docs/causal_mental_model.md)

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import matplotlib.pyplot as plt
import pandas as pd

from src.plotting import (
    COLOR_COUNTERFACTUAL,
    COLOR_TREATED,
    save_summary_figure,
    setup_style,
)
from src.causal.did import estimate_did
from src.causal.propensity import estimate_ipw_att
from src.causal.simulate import SynthSimulationConfig, simulate_heterogeneous_load
from src.causal.synth import (
    daily_peak_series,
    donor_pool_sensitivity,
    fit_synthetic_control,
    gap_uncertainty,
    in_space_placebos,
    in_time_placebos,
    inject_peak_reduction,
    naive_before_after_series,
    pre_period_sensitivity,
)
from src.data.pecan_street import PecanStreetNotAvailableError, load_pecan_street

setup_style()

## 1. Question

Did a peak-shaving tariff reduce **evening peak load** for treated homes, relative to what those homes would have used without the tariff?

That is a **counterfactual** question. Machine learning can predict next week's load; it cannot say what load would have been under a policy that did not happen. The object of interest is the average treatment effect on the treated (ATT) during peak hours (17:00–20:00).

A utility rolling this out cares about:

- the **kW (and %) cut** at the system peak
- whether that cut is **distinguishable from noise**
- whether the estimate is **stable** to reasonable analysis choices

## 2. Method

The pilot is **one treated unit** (the mean of five Pecan-style homes) plus many untreated donors. Synthetic control (Abadie) fits simplex weights on the **pre-tariff** window so a convex combination of donors tracks the treated path. Those same weights, applied after rollout, are the counterfactual `Y(0)`.

| Estimator | What it compares | Failure mode here |
|---|---|---|
| Naive before/after | Treated homes vs themselves over time | Shared weather jump is mistaken for the tariff |
| Difference-in-differences | Treated vs control, pre vs post | Needs parallel trends and two sizable groups |
| **Synthetic control** | Treated vs a weighted donor twin | Needs a donor pool that can match the pre-path |

**Placebos** ask whether a fake treatment produces a gap this large. **In-space:** pretend each untreated home was treated. **In-time:** pretend the tariff started on a date when nothing happened.

This run uses a **simulated** panel with a **known 15% peak-load cut** so we can check recovery. Pecan Street Dataport is the intended real source; it is not wired yet.

## 3. Experiment

Try Pecan Street; fall back to a heterogeneous simulator. Treatment is injected only on treated homes, post day 60, peak hours. A shared weather jump hits **everyone** after the same date — that is the confounder naive before/after will swallow.

In [ ]:
REDUCTION_PCT = 0.15
config = SynthSimulationConfig(seed=7)
intervention_day = config.n_days_before

try:
    raw = load_pecan_street()
    raise NotImplementedError("Real Pecan pipeline not wired yet; using simulator.")
except PecanStreetNotAvailableError:
    print("Pecan Street unavailable — using simulated heterogeneous panel.")
    panel = simulate_heterogeneous_load(config)

treated_ids = [f"PS{i:04d}" for i in range(config.n_treated)]
donor_ids = [f"PS{i + 100:04d}" for i in range(config.n_donors)]

panel = inject_peak_reduction(
    panel,
    treated_ids=treated_ids,
    intervention_day=intervention_day,
    reduction_pct=REDUCTION_PCT,
    peak_hours=config.peak_hours,
)

treated, donors = daily_peak_series(
    panel,
    treated_ids=treated_ids,
    donor_ids=donor_ids,
    peak_hours=config.peak_hours,
)
print(
    f"{len(treated)} days, {donors.shape[1]} donors, "
    f"{config.n_treated} treated homes, intervention on day {intervention_day}"
)

## 4. Counterfactual plot

Weights are fit only on pre-tariff days. After the vertical line, the dashed path is what treated peak load would have been without the tariff. The post-date gap is the effect.

In [ ]:
result = fit_synthetic_control(treated, donors, intervention_day=intervention_day)
uncertainty = gap_uncertainty(result, n_boot=500, seed=7)
naive = naive_before_after_series(treated, intervention_day=intervention_day)

top_weights = result.weights[result.weights > 0.01].sort_values(ascending=False)
print("Top donor weights:")
print(top_weights.to_string())
print(f"\nPre-period RMSPE: {result.pre_rmspe:.4f} kW")

fig, ax = plt.subplots(figsize=(12, 4.5))
days = result.treated.index
ax.plot(days, result.treated, label="Treated actual", color=COLOR_TREATED, linewidth=2)
ax.plot(
    days,
    result.synthetic,
    label="Synthetic counterfactual",
    color=COLOR_COUNTERFACTUAL,
    linewidth=2,
    linestyle="--",
)
ax.axvline(intervention_day, color="black", linestyle=":", linewidth=1.5, label="Tariff start")
ax.axvspan(intervention_day, days.max(), color=COLOR_TREATED, alpha=0.08)
ax.annotate(
    f"Post gap ≈ {result.att_kw:.2f} kW\n({result.att_pct:.1%})",
    xy=(
        intervention_day + 5,
        result.treated.loc[result.treated.index >= intervention_day].mean(),
    ),
    xytext=(intervention_day + 12, result.treated.max() - 0.1),
    arrowprops={"arrowstyle": "->", "color": "0.3"},
    fontsize=10,
)
ax.set_xlabel("Day")
ax.set_ylabel("Mean peak-hour load (kW)")
ax.set_title("Treated peak load vs synthetic Y(0)")
ax.legend(loc="upper left")
fig.tight_layout()
save_summary_figure(fig, "nb01_treated_vs_synthetic")
plt.close(fig)

## 5. Effect ± uncertainty

The ATT is the mean post-period gap (actual − synthetic). The interval below bootstraps those daily gaps while **holding the synthetic path fixed** — it captures day-to-day gap noise, not weight-estimation uncertainty. Placebo tests in the next section play that second role.

Recovery check: the simulator injected a **15%** peak cut. The SC percentage gap should land near that number. Naive before/after should not, because of the weather jump.

In [ ]:
print(
    f"Synthetic control ATT: {result.att_kw:.3f} kW "
    f"({result.att_pct:.1%} vs synthetic)"
)
print(
    f"95% bootstrap interval: [{uncertainty.ci_low:.3f}, {uncertainty.ci_high:.3f}] kW "
    f"(SE {uncertainty.std_error:.3f}, {uncertainty.n_post} post days)"
)
print(f"Injected reduction: {-REDUCTION_PCT:.1%}")
print(f"Recovery error (ATT% + 15 pp): {result.att_pct + REDUCTION_PCT:+.1%}")
print(f"Naive before/after gap: {naive:.3f} kW (confounded by weather)")

recovers = abs(result.att_pct + REDUCTION_PCT) < 0.04
excludes_zero = uncertainty.ci_high < 0.0
print(f"\nRecovers injected 15% (within 4 pp): {recovers}")
print(f"Bootstrap interval excludes zero: {excludes_zero}")

## 6. Placebo evidence

If the tariff effect is real, it should **stand out** from two null distributions:

- **In-space** — apply the same SC recipe to each untreated donor. Those homes never received the tariff.
- **In-time** — pick fake dates inside the true pre-period and estimate a 'post' gap on days that are known to be untreated.

The p-value is the share of placebo gaps at least as large as the treated gap (plus one, for the treated unit itself).

In [ ]:
space = in_space_placebos(treated, donors, intervention_day=intervention_day)
time = in_time_placebos(
    treated,
    donors,
    intervention_day=intervention_day,
    max_placebos=8,
)

print(f"In-space placebo p-value: {space.p_value:.3f}  ({len(space.placebo_gaps_kw)} donors)")
print(f"In-time placebo p-value:  {time.p_value:.3f}  ({len(time.placebo_gaps_kw)} fake dates)")
print(f"Treated gap: {space.treated_gap_kw:.3f} kW")
print(
    "In-space placebo range: "
    f"[{space.placebo_gaps_kw.min():.3f}, {space.placebo_gaps_kw.max():.3f}] kW"
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].hist(space.placebo_gaps_kw, bins=12, color="0.75", edgecolor="white")
axes[0].axvline(space.treated_gap_kw, color="C0", linewidth=2, label="Treated homes")
axes[0].set_xlabel("Post-period ATT (kW)")
axes[0].set_ylabel("Count")
axes[0].set_title("In-space: untreated donors as fake treated")
axes[0].legend()

axes[1].scatter(
    time.placebo_gaps_kw.index,
    time.placebo_gaps_kw.values,
    color="0.45",
    label="Fake dates",
    zorder=3,
)
axes[1].axhline(0.0, color="0.7", linewidth=1)
axes[1].axhline(
    time.treated_gap_kw,
    color="C0",
    linewidth=2,
    label="True tariff date",
)
axes[1].set_xlabel("Fake intervention day (true pre-period)")
axes[1].set_ylabel("Placebo ATT (kW)")
axes[1].set_title("In-time: fake dates before the real tariff")
axes[1].legend()

fig.tight_layout()
plt.show()

## 7. Cross-checks: DiD and propensity weighting

The same hourly panel can be read as a two-group experiment. **DiD** estimates `load ~ treated + post + treated:post` on peak hours — a parallel-trends cross-check, not a substitute for SC when the treated object is a single aggregate unit.

**IPW** reweights household *changes* in mean peak load by a logistic propensity on pre-period peak. In this simulator treatment is as-if random, so IPW is a method sketch for the opt-in case (high-usage homes self-selecting into a tariff). DoWhy's propensity-score weighting is tried if the library is importable.

In [ ]:
did = estimate_did(panel, peak_hours=config.peak_hours)
ipw = estimate_ipw_att(panel, peak_hours=config.peak_hours)

print(
    f"DiD ATT: {did.att_kw:.3f} kW  (SE {did.std_error:.3f}; "
    f"95% CI [{did.ci_low:.3f}, {did.ci_high:.3f}])"
)
print(
    f"IPW ATT (household peak change): {ipw.att_kw:.3f} kW  "
    f"[{ipw.n_treated} treated / {ipw.n_control} control]"
)

dowhy_att = None
try:
    from src.causal.propensity import estimate_dowhy_att

    dowhy = estimate_dowhy_att(panel, peak_hours=config.peak_hours)
    dowhy_att = dowhy.att_kw
    print(f"DoWhy propensity-score weighting ATT: {dowhy.att_kw:.3f} kW")
except Exception as exc:
    print(f"DoWhy path skipped ({type(exc).__name__}: {exc})")

comparison = pd.DataFrame(
    {
        "estimator": ["Synthetic control", "DiD", "IPW (pre-peak propensity)", "Naive before/after"],
        "att_kw": [result.att_kw, did.att_kw, ipw.att_kw, naive],
    }
)
if dowhy_att is not None:
    comparison = pd.concat(
        [comparison, pd.DataFrame({"estimator": ["DoWhy PS weighting"], "att_kw": [dowhy_att]})],
        ignore_index=True,
    )
print("\n" + comparison.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## 8. Robustness and recovery

Two analysis choices that often move SC estimates: **how many donors** remain in the pool, and **how long a pre-period** we use to fit weights. If the 15% recovery is real, ATT% should stay near −15% when we drop random donors or shorten the pre-window (down to a still-usable length).

The dashed line is the injected truth. Scatter around it is sensitivity, not a second placebo.

In [ ]:
donor_sens = donor_pool_sensitivity(
    treated,
    donors,
    intervention_day=intervention_day,
    drop_counts=(0, 5, 10, 15),
    n_draws=4,
    seed=7,
)
pre_sens = pre_period_sensitivity(
    treated,
    donors,
    intervention_day=intervention_day,
    pre_lengths=(20, 30, 40, 50, 60),
)

print("Donor-pool ATT% by donors kept:")
print(donor_sens.groupby("n_donors")["att_pct"].agg(["mean", "min", "max"]).to_string())
print("\nPre-period ATT% by pre-window length:")
print(pre_sens.set_index("pre_length")[["att_pct", "pre_rmspe"]].to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
axes[0].scatter(donor_sens["n_donors"], donor_sens["att_pct"], color="C0", alpha=0.7)
axes[0].axhline(-REDUCTION_PCT, color="0.3", linestyle="--", label="Injected −15%")
axes[0].set_xlabel("Donors kept")
axes[0].set_ylabel("ATT (share of synthetic)")
axes[0].set_title("Sensitivity: donor pool")
axes[0].legend()

axes[1].plot(pre_sens["pre_length"], pre_sens["att_pct"], marker="o", color="C1")
axes[1].axhline(-REDUCTION_PCT, color="0.3", linestyle="--", label="Injected −15%")
axes[1].set_xlabel("Pre-period days used to fit weights")
axes[1].set_title("Sensitivity: pre-period length")
axes[1].legend()

fig.tight_layout()
plt.show()

### What would break with a real (non-injected) tariff

This notebook **knows the answer** because we multiplied treated peak load by 0.85 after day 60. A real Pecan Street or Low Carbon London tariff does not come with that multiplier. Honest failure modes:

| Assumption | Simulator | Real meters |
|---|---|---|
| Treatment assignment | We chose the five IDs | Opt-in / eligibility; high-usage homes self-select |
| Donor contamination | Donors are untreated by construction | Neighbours, workplace load, or later enrolment leak |
| Effect timing | Sharp cut on a known day | Anticipation, delayed response, weekdays vs weekends |
| No interference | One home's tariff does not change another's load | Feeder-level voltage, social spillovers |
| Ground truth | 15% is known | Recovery check is **impossible**; placebos and pre-fit become the only diagnostics |
| Weather / season | One shared jump | Non-parallel climate, holidays, EV/PV uptake mid-sample |

If pre-period RMSPE is poor, if in-space placebos overlap the treated gap, or if shrinking the donor pool swings ATT by more than a few percentage points, **do not** treat the point estimate as rollout-ready — even if DiD agrees.

## 9. What this means for a utility deciding on rollout

Translate the ATT into a planning number, then read it with the uncertainty and placebos — not the point estimate alone.

Illustrative scaling (not a cost-benefit model): `peak MW avoided ≈ (−ATT_kW) × n_homes / 1000`. Wholesale value for one peak hour ≈ `MW × price_EUR_per_MWh`.

In [ ]:
N_PILOT = config.n_treated
N_ROLLOUT = 10_000
PRICE_EUR_PER_MWH = 120.0

pilot_mw = -result.att_kw * N_PILOT / 1000.0
rollout_mw = -result.att_kw * N_ROLLOUT / 1000.0
rollout_mw_low = -uncertainty.ci_high * N_ROLLOUT / 1000.0
rollout_mw_high = -uncertainty.ci_low * N_ROLLOUT / 1000.0

print(f"Pilot ({N_PILOT} homes): {pilot_mw:.4f} MW peak avoided")
print(
    f"If 10,000 similar homes enrolled: {rollout_mw:.2f} MW  "
    f"(interval {rollout_mw_low:.2f}–{rollout_mw_high:.2f} MW)"
)
print(
    f"One peak hour at €{PRICE_EUR_PER_MWH:.0f}/MWh: "
    f"€{rollout_mw * PRICE_EUR_PER_MWH:,.0f} "
    f"(interval €{rollout_mw_low * PRICE_EUR_PER_MWH:,.0f}–"
    f"€{rollout_mw_high * PRICE_EUR_PER_MWH:,.0f})"
)
print(
    "\nDecision read: the synthetic-control gap recovers the injected 15% cut, "
    "the bootstrap interval sits below zero, and both placebo batteries put the "
    "treated gap in the tail. That is evidence the *procedure* can detect a peak "
    "tariff of this size — not evidence that a real opt-in tariff will deliver 15%."
)

### Decision rule (this pilot, this notebook)

**On the simulated tariff:** the estimate is usable. SC, DiD, and IPW agree on a peak reduction; naive before/after does not (weather). Placebos say untreated homes and fake dates do not produce a gap this large. Sensitivity to donor count and pre-period length stays near the injected −15%.

**On a live rollout decision:** treat this as a **template**, not a result. Re-run on Pecan Street or Low Carbon London without an injected effect. Require (i) a tight pre-period match, (ii) treated gap extreme vs in-space and in-time placebos, (iii) a CI that excludes zero, and (iv) stability when the donor pool and pre-window move. If those fail, the honest recommendation is to **not** scale the tariff on this evidence — collect a cleaner assignment mechanism (randomized enrolment) or a larger donor pool, then estimate again.